In [ ]:
import pandas as pd

In [ ]:
chunk_list = []
chunk_size = 10000

json_reader = pd.read_json(
    'ossec-alerts-17.json',
    lines=True,
    chunksize=chunk_size
)

required_columns = ['rule', 'agent', 'manager', 'id']

for chunk in json_reader:

    missing_columns = [
        column for column in required_columns
        if column not in chunk.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    chunk_list.append(chunk[required_columns])

df = pd.concat(chunk_list, ignore_index=True)

print("Logs loaded successfully.")
print("Total logs:", len(df))
print("Columns:", list(df.columns))

In [ ]:
print(df.shape)

In [ ]:
print("Total logs loaded:", len(df))

In [ ]:
def extract_rule_level(rule):
    if isinstance(rule, dict):
        return rule.get('level', 0)
    return 0


def extract_rule_description(rule):
    if isinstance(rule, dict):
        return rule.get('description', 'Unknown')
    return 'Unknown'


def extract_agent_name(agent):
    if isinstance(agent, dict):
        return agent.get('name', 'Unknown')
    return 'Unknown'


def extract_agent_ip(agent):
    if isinstance(agent, dict):
        return agent.get('ip', 'Unknown')
    return 'Unknown'


df['Rule_Level'] = df['rule'].apply(extract_rule_level)
df['Rule_Description'] = df['rule'].apply(
    extract_rule_description
)

df['Agent_Name'] = df['agent'].apply(
    extract_agent_name
)

df['Agent_IP'] = df['agent'].apply(
    extract_agent_ip
)

print(
    df[
        [
            'Rule_Level',
            'Rule_Description',
            'Agent_Name',
            'Agent_IP'
        ]
    ].head()
)

In [ ]:
df_clean = df.drop(
    columns=['rule', 'agent', 'manager'],
    errors='ignore'
)

df_clean['Rule_Level'] = pd.to_numeric(
    df_clean['Rule_Level'],
    errors='coerce'
)

df_clean = df_clean.dropna(
    subset=['Rule_Level']
)

df_clean['Rule_Level'] = (
    df_clean['Rule_Level'].astype(int)
)

df_clean['Rule_Description'] = (
    df_clean['Rule_Description']
    .fillna('Unknown')
)

df_clean['Agent_Name'] = (
    df_clean['Agent_Name']
    .fillna('Unknown')
)

df_clean['Agent_IP'] = (
    df_clean['Agent_IP']
    .fillna('Unknown')
)

print("Clean dataset shape:", df_clean.shape)

In [ ]:
df_clean['Is_High_Severity'] = (
    df_clean['Rule_Level'] >= 10
).astype(int)

df_clean['Is_Login_Failure'] = (
    df_clean['Rule_Description']
    .str.contains(
        'Logon Failure',
        case=False,
        na=False
    )
).astype(int)

In [ ]:
print("========== DATA QUALITY REPORT ==========")

print(f"Total logs: {len(df_clean):,}")
print(f"Total columns: {len(df_clean.columns)}")

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nRule level statistics:")
print(df_clean['Rule_Level'].describe())

print(
    "\nHigh severity logs:",
    df_clean['Is_High_Severity'].sum()
)

print(
    "Login failure logs:",
    df_clean['Is_Login_Failure'].sum()
)

print("=========================================")

In [ ]:
# Create behavioral features

df_clean['Rule_Frequency'] = (
    df_clean['Rule_Description']
    .map(df_clean['Rule_Description'].value_counts())
)

df_clean['Agent_Event_Count'] = (
    df_clean['Agent_Name']
    .map(df_clean['Agent_Name'].value_counts())
)

features = [
    'Rule_Level',
    'Is_High_Severity',
    'Is_Login_Failure',
    'Rule_Frequency',
    'Agent_Event_Count'
]

X = df_clean[features]

print("ML dataset size:", len(X))
print("Features used:", features)

print("\nFeature summary:")
print(X.describe())

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.01,
    random_state=42
)

model.fit(X)

print("Model training complete!")

In [ ]:
predictions = model.predict(X)
scores = model.decision_function(X)

X_results = X.copy()

X_results['Anomaly_Label'] = predictions
X_results['Anomaly_Score'] = scores

anomalies = X_results[
    X_results['Anomaly_Label'] == -1
]

print("Total logs:", len(X_results))
print("Total anomalies found:", len(anomalies))

print(
    "Anomaly percentage:",
    round(len(anomalies) / len(X_results) * 100, 2),
    "%"
)

print("\nSample anomalies:")
print(anomalies.head())

In [ ]:
anomalies.to_csv('Detected_Threats.csv', index=False)
print("Threat report saved successfully!")

In [ ]:
df_clean.head(10000).to_csv('clean_logs.csv', index=False)

In [ ]:
import joblib

joblib.dump(model, 'isolation_forest_model.pkl')

print("Model saved successfully!")

In [ ]:
dashboard_data = df_clean[
    [
        'Rule_Level',
        'Is_High_Severity',
        'Is_Login_Failure',
        'Rule_Frequency',
        'Agent_Event_Count'
    ]
]

dashboard_data.to_csv(
    'processed_wazuh_logs.csv',
    index=False
)

print("Dashboard CSV created successfully!")
print("Rows:", len(dashboard_data))
print("Columns:", list(dashboard_data.columns))